Data Importer and Filter Notebook

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import openpyxl
import xlsxwriter
import random
from collections import defaultdict
from pathlib import Path
from datetime import datetime, timedelta
from abc import ABC, abstractmethod


# Set project folder as parent file; allow for data file to be pulled from data folder in notebook (more robust access method)
project_root = Path.cwd().parent
data_path = project_root / "data" / "AssetDB.csv"

# Background reference; general properties to be used throughout the code, update as needed

eu_exchanges = {"Xetra","London Stock Exchange","Deutsche Boerse AG","Euronext Amsterdam", "NASDAQ Helsinki","SIX Swiss Exchange","Euronext Paris","Borsa Italiana","Bolsas y Mercados Espanoles","Oslo Bors Asa"}

class Asset:

    n_assets = 0

    def __init__(self, ticker, name, asset_type, sector, exchange, currency, description):
        self.ticker = ticker
        self.name = name
        self.asset_type = asset_type
        self.sector = sector
        self.exchange = exchange
        self.currency = currency
        self.description = description
        self.region = "EU" if self.exchange in eu_exchanges else "US"

df = pd.read_csv(data_path)
companies = {row.stock:Asset(row.yfinance_ticker, row.name,row.asset_type, row.sector, row.exchange, row.currency, row.description) for row in df.itertuples()}

eu_assets = [a for a in companies.values() if a.region == "EU"]
us_assets = [a for a in companies.values() if a.region == "US"]

sectors = defaultdict(list)
for a in companies.values():
    sectors[a.sector].append(a)


In [ ]:
for sector in sectors:
    print(sector)

-
Computers, Phones & Household Electronics
Software & IT Services
Semiconductors & Semiconductor Equipment
Diversified Retail
Pharmaceuticals
Aerospace & Defense
Machinery, Equipment & Components
Consumer Goods Conglomerates
Oil & Gas
Oil & Gas Related Equipment and Services
Professional & Commercial Services
Communications & Networking
Automobiles & Auto Parts


In [39]:
today = datetime.today()
startdate = today - timedelta(days = 50)
print(str(today))
tickerlist = [a.ticker for a in companies.values()]
assetdata = yf.download(tickerlist, start = startdate, end = today)

print(assetdata)

2026-08-28 22:07:42.945539


[*********************100%***********************]  54 of 54 completed

Price            Close                                                 \
Ticker            AAPL   ABBNE.SW        ABBV      AIR.PA        AMZN   
Date                                                                    
2026-07-09         NaN  87.099998         NaN  196.520004         NaN   
2026-07-10  315.048309  87.099998  246.326675  197.259995  245.339996   
2026-07-13  317.036560  83.519997  246.247238  195.279999  247.309998   
2026-07-14  314.588684  85.419998  243.050003  195.940002  247.490005   
2026-07-15  327.217804  83.379997  244.110001  195.380005  254.960007   
2026-07-16  332.972839  79.080002  254.389999  195.619995  249.889999   
2026-07-17  333.452393  78.599998  254.490005  194.320007  247.229996   
2026-07-20  326.308563  79.680000  253.380005  191.779999  249.990005   
2026-07-21  327.457581  80.540001  256.100006  194.600006  247.550003   
2026-07-22  325.609192  81.000000  253.300003  208.300003  244.850006   
2026-07-23  321.382843  80.419998  256.920013  204.

In [59]:
# General "Portfolio" abstract class as blueprint for manual or automatic portfolio creation
class Portfolio(ABC):
    def __init__(self, name):
        self.name = name
        self.portfolio_assets=[]

    @abstractmethod
    def build(self):
        pass

# Manual Portfolio subclass to be filled in with a manual list of tickers
class Manual(Portfolio):
    def __init__(self, name):
        super().__init__(name)

    def build(self, assetlist):
        for asset in assetlist:
            if asset not in self.portfolio_assets:
                self.portfolio_assets.append(asset)

# Automated Portfolio subclass to be filled with region and/or sector weightings (or neither for random selection)
# Asset selection is still randomized for any type of portfolio (Random/Region Split/Region + Sector Split); worth considering separating this code
# so the portfolios don't have to be created on each run and can be stored separately from the main data import and data analysis.
class Automated(Portfolio):
    def __init__(self, name, weights=None):
        super().__init__(name)
        self.weights = weights

    def build(self, n = None):
        if self.weights is None:
            self.portfolio_assets = random.sample(list(companies.values()), n)
            return

        for region, split in self.weights.items():
            if isinstance(split,dict): # Region + Sector splits
                for sector, pct in split.items():
                    pool = [a for a in companies.values() if a.region == region and a.sector == sector]
                    count = round(n * pct)
                    self.portfolio_assets += random.sample(pool, min(count, len(pool)))
            else: # Region split only
                pool = [a for a in companies.values() if a.region == region]
                count = round(n * split)
                self.portfolio_assets += random.sample(pool, min(count, len(pool)))

Portfolio1 = Automated("Randomized Portfolio")
Portfolio1.build(40)

Portfolio2 = Automated("60/40", weights={"US":0.6,"EU":0.4})
Portfolio2.build(30)

for a in Portfolio2.portfolio_assets:
    print(str(a.ticker))

GE
MSFT
COP
MMM
SLB
NVDA
META
BA
AMZN
DE
EOG
MRK
JNJ
CVX
CRM
CAT
OXY
PFE
SAN.PA
AIR.PA
SAF.PA
SHEL.L
VOW3.DE
TTE.PA
BP.L
BAYN.DE
EXSA.DE
ABBNE.SW
VWCE.DE
NOVN.SW
